# 端到端模型优化样例

参考：[e2e_opt_model](https://github.com/apache/tvm/blob/main/docs/how_to/tutorials/e2e_opt_model.py)

本教程展示了如何使用 Apache TVM 来优化机器学习模型。使用来自 PyTorch 的预训练 ResNet-18 模型，并利用 TVM 的 Relax API 对其进行端到端的优化。请注意，默认的端到端优化可能不适合复杂的模型。

## 准备阶段

首先，准备模型和输入信息。使用来自PyTorch的预训练ResNet-18模型。

In [ ]:
import sys
sys.path.append("..")
import set_env

In [ ]:
import os
import numpy as np
import torch
from torch import fx
from torchvision.models.resnet import ResNet18_Weights, resnet18

torch_model = resnet18(weights=ResNet18_Weights.DEFAULT)

## 整体流程概述

```{figure} https://raw.githubusercontent.com/tlc-pack/web-data/main/images/design/tvm_overall_flow.svg
:align: center
:width: 80%
```

整体流程包括以下步骤：

- **构建或导入模型**：构建一个神经网络模型，或者从其他框架（如 PyTorch、ONNX）导入预训练的模型，并创建 TVM IRModule，其中包含编译所需的所有信息，包括用于计算图的高级别 Relax 函数和用于张量程序的低级 TensorIR 函数。
- **执行可组合优化**：执行一系列优化转换，例如图优化、张量程序优化和库调度。
- **构建和通用部署**：将优化后的模型构建为可在通用运行时部署的模块，并在不同设备上执行，如 CPU、GPU 或其他加速器。

## 将模型转换为 IRModule

使用 Relax 前端（面向 PyTorch）将模型转换为 IRModule，以便进一步优化。除了模型外，我们还需要提供输入的形状和数据类型。

In [ ]:
import tvm
from tvm import relax
from tvm.relax.frontend.torch import from_fx

torch_model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Give the input shape and data type
input_info = [((1, 3, 224, 224), "float32")]

# Convert the model to IRModule
with torch.no_grad():
    torch_fx_model = fx.symbolic_trace(torch_model)
    mod = from_fx(torch_fx_model, input_info, keep_params_as_input=True)

mod, params = relax.frontend.detach_params(mod)
mod.show()

## IRModule 优化

Apache TVM 提供了一个灵活的方式来优化 IRModule。围绕 IRModule 优化的所有运算都可以与现有的流水线组合。注意，每个转换都可以通过 ``tvm.ir.transform.Sequential`` 组合成一个优化流水线。

在本教程中，我们专注于通过自动调优（auto-tuning）对模型进行端到端优化。我们利用 MetaSchedule 调优模型，并将调优日志存储到数据库。我们还应用数据库到模型以获得最佳性能。

In [4]:
TOTAL_TRIALS = 8000  # Change to 20000 for better performance if needed
target = tvm.target.Target("nvidia/geforce-rtx-3090-ti")  # Change to your target device
work_dir = "tuning_logs"

# Skip running in CI environment
IS_IN_CI = os.getenv("CI", "") == "true"
if not IS_IN_CI:
    with target:
        mod = tvm.ir.transform.Sequential(
            [
                # Convert BatchNorm into a sequence of simpler ops for fusion
                relax.transform.DecomposeOpsForInference(),
                # Canonicalize the bindings
                relax.transform.CanonicalizeBindings(),
                # Run default optimization pipeline
                relax.get_pipeline("zero"),
                # Tune the model and store the log to database
                relax.transform.MetaScheduleTuneIRMod({}, work_dir, TOTAL_TRIALS),
                # Apply the database
                relax.transform.MetaScheduleApplyDatabase(work_dir),
            ]
        )(mod)

    # Only show the main function
    mod["main"].show()

2024-09-02 15:02:25 [INFO] [task_scheduler.cc:193] Sending 64 sample(s) to builder
2024-09-02 15:02:32 [INFO] [task_scheduler.cc:195] Sending 64 sample(s) to runner
2024-09-02 15:02:33 [INFO] [task_scheduler.cc:237] [Updated] Task #4: "fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4"


,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,fused_matmul_add13,1025000,1,N/A,N/A,N/A,192,
1,transpose,1,1,N/A,N/A,N/A,1,
2,reshape,1,1,N/A,N/A,N/A,5,
3,adaptive_avg_pool2d,25600,1,N/A,N/A,N/A,63,
4,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,231336448,1,N/A,N/A,N/A,192,
5,fused_conv2d_subtract_divide_expand_dims_multiply_expand_dims_add1_relu,240041984,1,N/A,N/A,N/A,64,
6,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1,232214528,2,N/A,N/A,N/A,64,
7,fused_conv2d8_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,115730944,1,N/A,N/A,N/A,128,
8,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_add12_relu4,231361536,2,N/A,N/A,N/A,64,
9,fused_conv2d3_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_relu2,231712768,1,N/A,N/A,N/A,64,


2024-09-02 15:02:33 [DEBUG] [task_scheduler.cc:318] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                    fused_matmul_add13 |   1025000 |      1 |            N/A |          N/A |                   N/A |    192 |      
  1 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      1 |      
  2 |                                                                               reshape |         1 |      1 |            N/A |          N/A |                   N/A |      5 |      
  3 |            

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,fused_matmul_add13,1025000,1,N/A,N/A,N/A,192,
1,transpose,1,1,N/A,N/A,N/A,1,
2,reshape,1,1,N/A,N/A,N/A,5,
3,adaptive_avg_pool2d,25600,1,N/A,N/A,N/A,63,
4,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,231336448,1,N/A,N/A,N/A,192,
5,fused_conv2d_subtract_divide_expand_dims_multiply_expand_dims_add1_relu,240041984,1,N/A,N/A,N/A,64,
6,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1,232214528,2,N/A,N/A,N/A,64,
7,fused_conv2d8_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,115730944,1,N/A,N/A,N/A,128,
8,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_add12_relu4,231361536,2,N/A,N/A,N/A,64,
9,fused_conv2d3_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_relu2,231712768,1,N/A,N/A,N/A,64,



Total trials: 0
Total latency (us): 0

2024-09-02 15:03:30 [DEBUG] [task_scheduler.cc:318] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                    fused_matmul_add13 |   1025000 |      1 |            N/A |          N/A |                   N/A |    192 |      
  1 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      1 |      
  2 |                                                                               reshape |         1 |      1 |            N/A |          N/A |                 

...

2024-09-02 15:19:13 [DEBUG] [task_scheduler.cc:318] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                    fused_matmul_add13 |   1025000 |      1 |            N/A |          N/A |                   N/A |    192 |      
  1 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      1 |      
  2 |                                                                               reshape |         1 |      1 |            N/A |          N/A |                   N/A |      5 |      
  3 |            

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,fused_matmul_add13,1025000,1,N/A,N/A,N/A,192,
1,transpose,1,1,N/A,N/A,N/A,1,
2,reshape,1,1,N/A,N/A,N/A,5,
3,adaptive_avg_pool2d,25600,1,N/A,N/A,N/A,63,
4,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,231336448,1,N/A,N/A,N/A,384,
5,fused_conv2d_subtract_divide_expand_dims_multiply_expand_dims_add1_relu,240041984,1,N/A,N/A,N/A,192,
6,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1,232214528,2,N/A,N/A,N/A,64,
7,fused_conv2d8_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,115730944,1,N/A,N/A,N/A,256,
8,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_add12_relu4,231361536,2,N/A,N/A,N/A,128,
9,fused_conv2d3_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_relu2,231712768,1,N/A,N/A,N/A,191,


2024-09-02 15:20:19 [DEBUG] [task_scheduler.cc:318] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                    fused_matmul_add13 |   1025000 |      1 |            N/A |          N/A |                   N/A |    192 |      
  1 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      1 |      
  2 |                                                                               reshape |         1 |      1 |            N/A |          N/A |                   N/A |      5 |      
  3 |            

,Name,FLOP,Weight,Speed (GFLOPS),Latency (us),Weighted Latency (us),Trials,Done
0,fused_matmul_add13,1025000,1,N/A,N/A,N/A,192,
1,transpose,1,1,N/A,N/A,N/A,1,
2,reshape,1,1,N/A,N/A,N/A,5,
3,adaptive_avg_pool2d,25600,1,N/A,N/A,N/A,63,
4,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,231336448,1,N/A,N/A,N/A,384,
5,fused_conv2d_subtract_divide_expand_dims_multiply_expand_dims_add1_relu,240041984,1,N/A,N/A,N/A,192,
6,fused_conv2d1_subtract1_divide1_expand_dims_multiply1_expand_dims_add2_relu1,232214528,2,N/A,N/A,N/A,64,
7,fused_conv2d8_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_relu4,115730944,1,N/A,N/A,N/A,256,
8,fused_conv2d9_subtract4_divide4_expand_dims3_multiply4_expand_dims3_add11_add12_relu4,231361536,2,N/A,N/A,N/A,128,
9,fused_conv2d3_subtract2_divide2_expand_dims1_multiply2_expand_dims1_add5_relu2,231712768,1,N/A,N/A,N/A,191,


2024-09-02 15:21:20 [DEBUG] [task_scheduler.cc:318] 
 ID |                                                                                  Name |      FLOP | Weight | Speed (GFLOPS) | Latency (us) | Weighted Latency (us) | Trials | Done 
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  0 |                                                                    fused_matmul_add13 |   1025000 |      1 |            N/A |          N/A |                   N/A |    192 |      
  1 |                                                                             transpose |         1 |      1 |            N/A |          N/A |                   N/A |      1 |      
  2 |                                                                               reshape |         1 |      1 |            N/A |          N/A |                   N/A |      5 |      
  3 |            

## 构建和部署

最后，我们构建优化后的模型并将其部署到目标设备。

In [ ]:
if not IS_IN_CI:
    ex = relax.build(mod, target="cuda")
    dev = tvm.device("cuda", 0)
    vm = relax.VirtualMachine(ex, dev)
    # Need to allocate data and params on GPU device
    gpu_data = tvm.nd.array(np.random.rand(1, 3, 224, 224).astype("float32"), dev)
    gpu_params = [tvm.nd.array(p, dev) for p in params["main"]]
    gpu_out = vm["main"](gpu_data, *gpu_params).numpy()

    print(gpu_out.shape)